# Notebook 05 - Jobs, orchestration et monitoring

## Objectif

Un pipeline réel ne s'arrête pas au chargement des tables.  
Il faut aussi pouvoir répondre à des questions d'exploitation :

- quel job s'exécute quand,
- dans quel ordre,
- avec quelles dépendances,
- combien de temps il prend,
- combien de lignes il lit,
- combien il rejette,
- s'il a nécessité un retry.

## Ce qu'on va produire

- `ops_job_definition`
- `ops_job_run`
- `ops_job_step_run`

In [1]:
# Nous importons pandas.
import pandas as pd

# Nous importons la bibliothèque pathlib pour manipuler les chemins de fichiers proprement.
from pathlib import Path

In [2]:
# Nous définissons un chemin de départ qui correspond au dossier courant du notebook.
current_dir = Path.cwd().resolve()

# Si le notebook est exécuté depuis le dossier du projet, ce test sera vrai.
if (current_dir / "data").exists():
    # Dans ce cas, le dossier courant est déjà la racine du projet.
    project_root = current_dir
else:
    # Sinon, on suppose que le notebook est exécuté depuis le dossier notebooks.
    project_root = current_dir.parent

# Nous définissons ici le dossier qui contient les lots bruts.
data_dir = project_root / "data" / "raw_batches"

# Nous définissons ici le dossier de sortie.
output_dir = project_root / "outputs"

# Nous affichons la racine retenue pour éviter toute ambiguïté.
print("project_root =", project_root)

# Nous affichons la liste des fichiers bruts disponibles.
print("raw files =", sorted(p.name for p in data_dir.glob("*.json")))

project_root = D:\Ens-EPSI\boamp_etl_dwh_tiny_explained_project
raw files = ['B20150308_01_initial.json', 'B20150309_01_incremental.json', 'B20150309_01_rerun.json']


## Étape 1 - Charger la table de contrôle des batchs

Cette table nous sert de base pour simuler les exécutions.

In [3]:
# Nous lisons la table de batch control.
ops_batch_control = pd.read_csv(output_dir / "staging" / "ops_batch_control.csv")

# Nous convertissons la date d'extraction en timestamp UTC.
ops_batch_control["extraction_ts"] = pd.to_datetime(ops_batch_control["extraction_ts"], utc=True)

# Nous affichons la table.
ops_batch_control

,batch_id,batch_type,source_system,extraction_ts,received_records,status
0,B20150308_01_initial,INITIAL,BOAMP_API,2015-03-08 05:30:00+00:00,4,SUCCESS
1,B20150309_01_incremental,INCREMENTAL,BOAMP_API,2015-03-09 05:30:00+00:00,3,SUCCESS
2,B20150309_01_rerun,RERUN,BOAMP_API,2015-03-09 06:00:00+00:00,3,SUCCESS


## Étape 2 - Définir les jobs

Nous restons simples et réalistes.

### Jobs 

1. `collect_boamp_batch`
2. `stage_boamp_notice`
3. `transform_notice_kpi`
4. `load_notice_dwh`
5. `refresh_kpi_tables`

L'ordre des jobs est critique. 


In [4]:
# Nous créons la table de définition des jobs.
ops_job_definition = pd.DataFrame(
    [
        {
            "job_name": "collect_boamp_batch",
            "layer": "collect",
            "schedule": "05:30 daily",
            "depends_on": None,
            "sla_target": "05:35",
        },
        {
            "job_name": "stage_boamp_notice",
            "layer": "staging",
            "schedule": "05:35 daily",
            "depends_on": "collect_boamp_batch",
            "sla_target": "05:40",
        },
        {
            "job_name": "transform_notice_kpi",
            "layer": "transform",
            "schedule": "05:40 daily",
            "depends_on": "stage_boamp_notice",
            "sla_target": "05:45",
        },
        {
            "job_name": "load_notice_dwh",
            "layer": "dwh",
            "schedule": "05:45 daily",
            "depends_on": "transform_notice_kpi",
            "sla_target": "05:50",
        },
        {
            "job_name": "refresh_kpi_tables",
            "layer": "bi",
            "schedule": "05:50 daily",
            "depends_on": "load_notice_dwh",
            "sla_target": "05:55",
        },
    ]
)

# Nous affichons la définition des jobs.
ops_job_definition

,job_name,layer,schedule,depends_on,sla_target
0,collect_boamp_batch,collect,05:30 daily,NaN,05:35
1,stage_boamp_notice,staging,05:35 daily,collect_boamp_batch,05:40
2,transform_notice_kpi,transform,05:40 daily,stage_boamp_notice,05:45
3,load_notice_dwh,dwh,05:45 daily,transform_notice_kpi,05:50
4,refresh_kpi_tables,bi,05:50 daily,load_notice_dwh,05:55


## Étape 3 - Simuler les exécutions de jobs

Dans un vrai orchestrateur, ces données viendraient d'Airflow ou autre.  
Ici, on les simule pour montrer ce qu'on suit réellement.

In [5]:
# Nous préparons deux listes vides.
job_run_rows = []
job_step_rows = []

# Nous initialisons un compteur de run.
run_counter = 1

# Nous parcourons les batchs reçus.
for batch in ops_batch_control.to_dict("records"):

    # Pour chaque batch, nous simulons l'exécution des cinq jobs.
    for job_name, duration_min in [
        ("collect_boamp_batch", 2),
        ("stage_boamp_notice", 2),
        ("transform_notice_kpi", 1),
        ("load_notice_dwh", 1),
        ("refresh_kpi_tables", 1),
    ]:

        # Nous construisons un identifiant de run.
        run_id = f"RUN_{run_counter:03d}"

        # Nous prenons comme heure de départ l'heure d'extraction du batch.
        start_ts = batch["extraction_ts"]

        # Nous calculons une heure de fin simple.
        end_ts = start_ts + pd.Timedelta(minutes=duration_min)

        # Nous simulons un nombre de lignes lues égal au volume reçu.
        rows_read = batch["received_records"]

        # Nous simulons un rejet sur le batch incrémental au moment du staging.
        rows_rejected = 1 if batch["batch_type"] == "INCREMENTAL" and job_name == "stage_boamp_notice" else 0

        # Nous simulons un retry sur le rerun pendant la collecte.
        retry_count = 1 if batch["batch_type"] == "RERUN" and job_name == "collect_boamp_batch" else 0

        # Nous ajoutons la ligne de run.
        job_run_rows.append(
            {
                "run_id": run_id,
                "job_name": job_name,
                "batch_id": batch["batch_id"],
                "start_ts": start_ts,
                "end_ts": end_ts,
                "duration_seconds": int((end_ts - start_ts).total_seconds()),
                "status": "SUCCESS",
                "retry_count": retry_count,
                "rows_read": rows_read,
                "rows_rejected": rows_rejected,
            }
        )

        # Nous ajoutons aussi une ligne de step pour illustrer le suivi fin.
        job_step_rows.append(
            {
                "run_id": run_id,
                "step_name": "main_step",
                "status": "SUCCESS",
                "rows_read": rows_read,
                "rows_rejected": rows_rejected,
            }
        )

        # Nous incrémentons le compteur.
        run_counter += 1

# Nous créons les DataFrames.
ops_job_run = pd.DataFrame(job_run_rows)
ops_job_step_run = pd.DataFrame(job_step_rows)

# Nous affichons les résultats.
display(ops_job_run)
display(ops_job_step_run)


,run_id,job_name,batch_id,start_ts,end_ts,duration_seconds,status,retry_count,rows_read,rows_rejected
0,RUN_001,collect_boamp_batch,B20150308_01_initial,2015-03-08 05:30:00+00:00,2015-03-08 05:32:00+00:00,120,SUCCESS,0,4,0
1,RUN_002,stage_boamp_notice,B20150308_01_initial,2015-03-08 05:30:00+00:00,2015-03-08 05:32:00+00:00,120,SUCCESS,0,4,0
2,RUN_003,transform_notice_kpi,B20150308_01_initial,2015-03-08 05:30:00+00:00,2015-03-08 05:31:00+00:00,60,SUCCESS,0,4,0
3,RUN_004,load_notice_dwh,B20150308_01_initial,2015-03-08 05:30:00+00:00,2015-03-08 05:31:00+00:00,60,SUCCESS,0,4,0
4,RUN_005,refresh_kpi_tables,B20150308_01_initial,2015-03-08 05:30:00+00:00,2015-03-08 05:31:00+00:00,60,SUCCESS,0,4,0
5,RUN_006,collect_boamp_batch,B20150309_01_incremental,2015-03-09 05:30:00+00:00,2015-03-09 05:32:00+00:00,120,SUCCESS,0,3,0
6,RUN_007,stage_boamp_notice,B20150309_01_incremental,2015-03-09 05:30:00+00:00,2015-03-09 05:32:00+00:00,120,SUCCESS,0,3,1
7,RUN_008,transform_notice_kpi,B20150309_01_incremental,2015-03-09 05:30:00+00:00,2015-03-09 05:31:00+00:00,60,SUCCESS,0,3,0
8,RUN_009,load_notice_dwh,B20150309_01_incremental,2015-03-09 05:30:00+00:00,2015-03-09 05:31:00+00:00,60,SUCCESS,0,3,0
9,RUN_010,refresh_kpi_tables,B20150309_01_incremental,2015-03-09 05:30:00+00:00,2015-03-09 05:31:00+00:00,60,SUCCESS,0,3,0


,run_id,step_name,status,rows_read,rows_rejected
0,RUN_001,main_step,SUCCESS,4,0
1,RUN_002,main_step,SUCCESS,4,0
2,RUN_003,main_step,SUCCESS,4,0
3,RUN_004,main_step,SUCCESS,4,0
4,RUN_005,main_step,SUCCESS,4,0
5,RUN_006,main_step,SUCCESS,3,0
6,RUN_007,main_step,SUCCESS,3,1
7,RUN_008,main_step,SUCCESS,3,0
8,RUN_009,main_step,SUCCESS,3,0
9,RUN_010,main_step,SUCCESS,3,0


## Étape 4 - Sauvegarder les tables ops

In [6]:
# Nous définissons le dossier ops.
ops_dir = output_dir / "ops"

# Nous créons le dossier si besoin.
ops_dir.mkdir(parents=True, exist_ok=True)

# Nous écrivons la définition des jobs.
ops_job_definition.to_csv(ops_dir / "ops_job_definition.csv", index=False)

# Nous écrivons l'historique des runs.
ops_job_run.to_csv(ops_dir / "ops_job_run.csv", index=False)

# Nous écrivons l'historique des steps.
ops_job_step_run.to_csv(ops_dir / "ops_job_step_run.csv", index=False)

# Nous affichons les fichiers créés.
sorted(p.name for p in ops_dir.glob("*.csv"))

['ops_job_definition.csv', 'ops_job_run.csv', 'ops_job_step_run.csv']

### Un job n'est pas "juste du code"

C'est aussi :

- un horaire,
- une dépendance,
- un statut,
- une durée,
- un volume,
- un nombre de rejets,
- un éventuel retry.

### Pourquoi c'est important

Parce qu'un pipeline peut être "SUCCESS" techniquement tout en étant mauvais :

- trop lent,
- trop rejeté,
- vide,
- en retard par rapport au SLA (Service Level Agreement).